# 02 - Orders First-Pass Cleaning

This notebook records the evidence, decisions, and reasons used to prepare the Olist orders table for delivery analysis. The raw CSV is never modified. The reusable implementation is in `../src/clean_orders.py`.

## Decision log

| Issue | Evidence | Decision | Reason |
|---|---|---|---|
| Duplicate rows and keys | 0 exact duplicate rows; 0 duplicate `order_id` values | Remove no rows | Every order is unique in the orders table |
| Date types | Five timestamp columns loaded as `object`; 0 non-null values failed conversion | Convert all five columns with `pd.to_datetime` | Datetime values are required for interval calculations |
| Missing delivery dates | 2,965 missing customer-delivery timestamps; most belong to non-delivered statuses | Keep all rows in the cleaned base table | Missing delivery dates can be structurally valid for canceled, unavailable, or unfinished orders |
| Delivery metric eligibility | 96,478 delivered orders; 96,470 have the required actual and estimated dates | Use 96,470 orders for delivery metrics; retain the other 8 with an ineligible flag | Orders without both dates cannot be classified as late or on time |
| Timestamp inconsistencies | 166 carrier timestamps precede purchase; 23 customer-delivery timestamps precede carrier timestamps | Retain and flag the records | Flags preserve evidence and allow metric-specific exclusions later |
| Extreme delay values | Observed `delay_days` range is -146.02 to 188.98 days | Retain the values for now | Severe delays are relevant to the business question and are not proven data-entry errors |

## Load the raw orders table

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw_file = project_root / 'data' / 'raw' / 'olist_orders_dataset.csv'
orders = pd.read_csv(raw_file)

## Check duplicates

In [ ]:
orders.duplicated().sum(), orders['order_id'].duplicated().sum()

## Convert dates and inspect missing values

In [ ]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column], errors='coerce')

orders[date_columns].isna().sum()

## Build delivery metrics without deleting ineligible orders

In [ ]:
eligible = (
    orders['order_status'].eq('delivered')
    & orders['order_delivered_customer_date'].notna()
    & orders['order_estimated_delivery_date'].notna()
)
orders['delivery_analysis_eligible'] = eligible
orders.loc[eligible, 'delay_days'] = (
    orders.loc[eligible, 'order_delivered_customer_date']
    - orders.loc[eligible, 'order_estimated_delivery_date']
).dt.total_seconds() / 86400
orders.loc[eligible, 'is_late'] = orders.loc[eligible, 'delay_days'] > 0

## Add anomaly flags

In [ ]:
orders['flag_carrier_before_purchase'] = (
    orders['order_delivered_carrier_date'] < orders['order_purchase_timestamp']
)
orders['flag_customer_before_carrier'] = (
    orders['order_delivered_customer_date'] < orders['order_delivered_carrier_date']
)

## Verified first-pass results

- Cleaned base table: 99,441 rows and 13 columns.
- Delivery-analysis eligible: 96,470 orders.
- Late orders: 7,826; initial late-delivery rate: 8.11%.
- Median `delay_days`: -11.95 days.
- No source rows were deleted.
- The local processed file is `data/processed/olist_orders_cleaned.csv` and is excluded from Git because it is generated data.